# Notebook 01 — Data Cleaning
### Persistent Racial Disparities in U.S. Mortgage Approval: Evidence from 42 Million Applications, 2020–2024

**Author:** Rajveer Singh Pall  
**Institution:** Gyan Ganga Institute of Technology and Sciences  

---

Processes raw HMDA microdata (2020–2024) into cleaned annual panel files using memory-efficient chunked I/O. Applies race-code validation, computes loan-to-value (LTV) and debt-to-income (DTI) ratios, and exports verification summaries.

**Input:** `data/raw/hmda_{year}.csv` (2020–2024)  
**Output:** `data/processed/panel_{year}.csv`, `data/processed/cleaning_summary.csv`  
**Runtime:** ~20 minutes per year



In [2]:
from pathlib import Path

# Create canonical output directories
Path("../outputs/tables").mkdir(parents=True, exist_ok=True)
Path("../outputs/figures").mkdir(parents=True, exist_ok=True)
print("✅ Output directories created")

✅ Output directories created


In [3]:
"""
NOTEBOOK 01: DATA CLEANING
==========================
Cleans raw HMDA data (2020-2024) into panel files

Implementation notes:
- Uses CORRECT race codes: 3=Black, 5=White (NOT 2!)
- Handles missing values properly
- Memory-efficient chunked processing

INPUT:  data/raw/hmda_2020.csv through hmda_2024.csv
OUTPUT: data/processed/panel_2020.csv through panel_2024.csv
        data/processed/cleaning_summary.csv

RUNTIME: ~20 minutes per year (100 min total for all 5 years)
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("HMDA DATA CLEANING PIPELINE")
print("="*70)
print("\n✅ Libraries loaded successfully")

HMDA DATA CLEANING PIPELINE

✅ Libraries loaded successfully


In [4]:
# ============================================================================
# CORRECT HMDA RACE CODES
# ============================================================================

BLACK_CODE = 3  # Black or African American (CORRECT)
WHITE_CODE = 5  # White (CORRECT)

print("RACE CODE CONFIGURATION:")
print(f"  Black code: {BLACK_CODE}")
print(f"  White code: {WHITE_CODE}")
print("\n⚠️  IMPORTANT: We are using codes 3 and 5, NOT 2!")

# ============================================================================
# FILE PATHS
# ============================================================================

RAW_DATA_DIR = Path("../data")
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nDIRECTORIES:")
print(f"  Raw data: {RAW_DATA_DIR}")
print(f"  Processed data: {PROCESSED_DATA_DIR}")

# ============================================================================
# PROCESSING PARAMETERS
# ============================================================================

CHUNK_SIZE = 200_000  # Process 200K rows at a time (memory efficient)
YEARS = [2020, 2021, 2022, 2023, 2024]

print(f"\nPROCESSING SETTINGS:")
print(f"  Chunk size: {CHUNK_SIZE:,} rows")
print(f"  Years to process: {YEARS}")

# ============================================================================
# COLUMNS TO KEEP
# ============================================================================

KEEP_COLS = [
    "lei",                    # Lender identifier
    "activity_year",          # Year
    "state_code",            # State code (2 letters)
    "action_taken",          # Approval decision (1=approved, 3=denied)
    "applicant_race_1",      # Race code (3=Black, 5=White)
    "income",                # Applicant income ($1000s)
    "loan_amount",           # Loan amount ($1000s)
    "property_value",        # Property value ($1000s)
    "interest_rate",         # Interest rate (%)
    "rate_spread",           # Rate spread above benchmark (%)
    "loan_purpose"
]

print(f"\nCOLUMNS TO EXTRACT: {len(KEEP_COLS)}")
for i, col in enumerate(KEEP_COLS, 1):
    print(f"  {i:2d}. {col}")

RACE CODE CONFIGURATION:
  Black code: 3
  White code: 5

⚠️  IMPORTANT: We are using codes 3 and 5, NOT 2!

DIRECTORIES:
  Raw data: ..\data
  Processed data: ..\data\processed

PROCESSING SETTINGS:
  Chunk size: 200,000 rows
  Years to process: [2020, 2021, 2022, 2023, 2024]

COLUMNS TO EXTRACT: 11
   1. lei
   2. activity_year
   3. state_code
   4. action_taken
   5. applicant_race_1
   6. income
   7. loan_amount
   8. property_value
   9. interest_rate
  10. rate_spread
  11. loan_purpose


In [5]:
def clean_hmda_chunk(chunk):
    """
    Clean a single chunk of HMDA data.
    
    Applies filters:
    1. Keep only approved (1) or denied (3) applications
    2. Keep only Black (3) or White (5) applicants
    3. Drop missing LEI (lender identifier)
    4. Drop missing income, loan_amount, or property_value
    
    Creates derived variables:
    - approved: 1 if action_taken=1, 0 if action_taken=3
    - black: 1 if applicant_race_1=3, 0 if applicant_race_1=5
    
    Returns:
        Cleaned DataFrame or None if empty after filtering
    """
    # Keep only needed columns
    available_cols = [c for c in KEEP_COLS if c in chunk.columns]
    chunk = chunk[available_cols].copy()
    
    # FILTER 1: Keep only approved (1) or denied (3)
    chunk = chunk[chunk["action_taken"].isin([1, 3])]
    if len(chunk) == 0:
        return None
    
    # FILTER 2: Keep only Black (3) or White (5) - CORRECT CODES
    if "applicant_race_1" in chunk.columns:
        chunk = chunk[chunk["applicant_race_1"].isin([BLACK_CODE, WHITE_CODE])]
    if len(chunk) == 0:
        return None
    
    # FILTER 3: Drop missing LEI
    if "lei" in chunk.columns:
        chunk = chunk[chunk["lei"].notna()]
    if len(chunk) == 0:
        return None
    
    # FILTER 4: Drop missing critical variables
    critical = ["income", "loan_amount", "property_value"]
    critical_present = [c for c in critical if c in chunk.columns]
    if critical_present:
        chunk = chunk.dropna(subset=critical_present)
    if len(chunk) == 0:
        return None
    
    # Create binary approval indicator
    # approved = 1 if loan originated (action_taken=1)
    # approved = 0 if denied (action_taken=3)
    chunk["approved"] = pd.NA
    chunk.loc[chunk["action_taken"] == 1, "approved"] = 1
    chunk.loc[chunk["action_taken"] == 3, "approved"] = 0
    
    # Create Black indicator (CORRECT CODE 3)
    chunk["black"] = (chunk["applicant_race_1"] == BLACK_CODE).astype(int)
    
    return chunk

print("✅ Function defined: clean_hmda_chunk()")

✅ Function defined: clean_hmda_chunk()


In [6]:
def verify_panel_file(filepath):
    """
    Verify a panel file has correct structure and values.
    
    Checks:
    - Race codes are only 3 and 5
    - Black indicator matches race code
    - Approval values are only 0 and 1
    """
    print(f"\n{'─'*70}")
    print(f"VERIFYING: {filepath.name}")
    print(f"{'─'*70}")
    
    # Read first 100K rows to check
    sample = pd.read_csv(filepath, nrows=100_000)
    
    # Check 1: Race codes
    race_codes = sorted(sample["applicant_race_1"].unique())
    print(f"\n1. Race codes present: {race_codes}")
    
    if set(race_codes) == {BLACK_CODE, WHITE_CODE}:
        print(f"   ✅ CORRECT: Only codes {BLACK_CODE} (Black) and {WHITE_CODE} (White)")
    else:
        print(f"   ❌ ERROR: Expected only {BLACK_CODE} and {WHITE_CODE}")
        return False
    
    # Check 2: Black indicator matches race code
    black_from_code = (sample["applicant_race_1"] == BLACK_CODE).sum()
    black_from_indicator = (sample["black"] == 1).sum()
    
    print(f"\n2. Black indicator consistency:")
    print(f"   From race code 3: {black_from_code:,}")
    print(f"   From black=1:     {black_from_indicator:,}")
    
    if black_from_code == black_from_indicator:
        print(f"   ✅ MATCH: Black indicator is correct")
    else:
        print(f"   ❌ MISMATCH: Black indicator is wrong!")
        return False
    
    # Check 3: Approval values
    approval_vals = sorted(sample["approved"].dropna().unique())
    print(f"\n3. Approval values: {approval_vals}")
    
    if set(approval_vals) == {0, 1}:
        print(f"   ✅ CORRECT: Only 0 (denied) and 1 (approved)")
    else:
        print(f"   ❌ ERROR: Expected only 0 and 1")
        return False
    
    # Summary statistics
    print(f"\n4. Summary (first 100K rows):")
    print(f"   Black share:     {sample['black'].mean()*100:.1f}%")
    print(f"   Approval rate:   {sample['approved'].mean()*100:.1f}%")
    print(f"   Missing income:  {sample['income'].isna().sum():,}")
    print(f"   Missing rate:    {sample['interest_rate'].isna().sum():,}")
    
    print(f"\n{'─'*70}")
    print("✅ VERIFICATION PASSED")
    print(f"{'─'*70}")
    
    return True

print("✅ Function defined: verify_panel_file()")

✅ Function defined: verify_panel_file()


In [7]:
def process_year(year):
    """
    Process one year of HMDA data.

    Steps:
    1. Read raw HMDA file in chunks
    2. Clean each chunk
    3. Compute LTV and DTI safely
    4. Write to panel file
    5. Verify output
    """

    print(f"\n{'='*70}")
    print(f"PROCESSING YEAR {year}")
    print(f"{'='*70}")

    input_file = RAW_DATA_DIR / f"hmda_{year}.csv"
    output_file = PROCESSED_DATA_DIR / f"panel_{year}.csv"

    if not input_file.exists():
        print(f"❌ ERROR: Input file not found: {input_file}")
        return None

    print(f"\nInput:  {input_file}")
    print(f"Output: {output_file}")

    # Counters
    chunks_processed = 0
    total_rows_kept = 0
    total_black = 0
    total_white = 0
    total_approved = 0
    total_denied = 0

    first_write = True

    try:
        reader = pd.read_csv(
            input_file,
            chunksize=CHUNK_SIZE,
            engine="python",
            on_bad_lines="skip"
        )

        print(f"\nProcessing in chunks of {CHUNK_SIZE:,} rows...")

        for i, chunk in enumerate(reader):

            cleaned = clean_hmda_chunk(chunk)
            if cleaned is None or len(cleaned) == 0:
                continue

            # Add year
            cleaned["year"] = year

            # -------------------------------------------------
            # FORCE NUMERIC TYPES 
            # -------------------------------------------------
            numeric_cols = [
                "income",
                "loan_amount",
                "property_value",
                "interest_rate"
            ]

            for col in numeric_cols:
                if col in cleaned.columns:
                    cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

            # -------------------------------------------------
            # LOAN-TO-VALUE (LTV)
            # -------------------------------------------------
            cleaned["ltv"] = (cleaned["loan_amount"] / cleaned["property_value"]) * 100
            cleaned.loc[
                (cleaned["ltv"] <= 0) | (cleaned["ltv"] > 200),
                "ltv"
            ] = pd.NA

            # -------------------------------------------------
            # DEBT-TO-INCOME (DTI)
            # -------------------------------------------------
            r_annual = cleaned["interest_rate"] / 100
            r_monthly = r_annual / 12
            n_months = 360

            monthly_payment = np.where(
                r_monthly > 0,
                cleaned["loan_amount"]
                * r_monthly
                * (1 + r_monthly) ** n_months
                / ((1 + r_monthly) ** n_months - 1),
                cleaned["loan_amount"] / n_months
            )

            cleaned["dti"] = (monthly_payment / (cleaned["income"] / 12)) * 100
            cleaned.loc[
                (cleaned["dti"] <= 0) | (cleaned["dti"] > 100),
                "dti"
            ] = pd.NA

            # -------------------------------------------------
            # FINAL COLUMN ORDER
            # -------------------------------------------------
            panel_cols = [
                "lei", "year", "state_code", "applicant_race_1",
                "black", "approved", "income", "loan_amount",
                "property_value", "interest_rate", "rate_spread",
                "ltv", "dti", "loan_purpose"
            ]

            cleaned = cleaned[[c for c in panel_cols if c in cleaned.columns]]

            # Write chunk
            cleaned.to_csv(
                output_file,
                mode="w" if first_write else "a",
                index=False,
                header=first_write
            )
            first_write = False

            # Update counters
            chunks_processed += 1
            total_rows_kept += len(cleaned)
            total_black += (cleaned["black"] == 1).sum()
            total_white += (cleaned["black"] == 0).sum()
            total_approved += (cleaned["approved"] == 1).sum()
            total_denied += (cleaned["approved"] == 0).sum()

            if (i + 1) % 10 == 0:
                print(f"  Chunk {i+1}: {(i+1) * CHUNK_SIZE:,} rows processed...")

    except Exception as e:
        print(f"\n❌ ERROR processing year {year}: {e}")
        import traceback
        traceback.print_exc()
        return None

    # -------------------------------------------------
    # SUMMARY
    # -------------------------------------------------
    print(f"\n{'─'*70}")
    print(f"YEAR {year} SUMMARY")
    print(f"{'─'*70}")
    print(f"Chunks processed:    {chunks_processed:,}")
    print(f"Total rows kept:     {total_rows_kept:,}")
    print(f"Black applicants:    {total_black:,} ({total_black/total_rows_kept*100:.1f}%)")
    print(f"White applicants:    {total_white:,} ({total_white/total_rows_kept*100:.1f}%)")
    print(f"Approved:            {total_approved:,} ({total_approved/total_rows_kept*100:.1f}%)")
    print(f"Denied:              {total_denied:,} ({total_denied/total_rows_kept*100:.1f}%)")
    print(f"✅ Saved to: {output_file}")

    verify_panel_file(output_file)

    return {
        "year": year,
        "total_rows": total_rows_kept,
        "black_count": total_black,
        "white_count": total_white,
        "black_pct": total_black / total_rows_kept * 100,
        "approved_count": total_approved,
        "denied_count": total_denied,
        "approval_rate": total_approved / total_rows_kept * 100
    }




In [8]:
# Process all years and collect results
print("="*70)
print("STARTING DATA CLEANING FOR ALL YEARS")
print("="*70)
print("\nThis will take approximately 100 minutes (20 min per year)")
print("You can monitor progress below...\n")

results = []

for year in YEARS:
    result = process_year(year)
    if result is not None:
        results.append(result)

print("\n" + "="*70)
print("ALL YEARS PROCESSED")
print("="*70)

STARTING DATA CLEANING FOR ALL YEARS

This will take approximately 100 minutes (20 min per year)
You can monitor progress below...


PROCESSING YEAR 2020

Input:  ..\data\hmda_2020.csv
Output: ..\data\processed\panel_2020.csv

Processing in chunks of 200,000 rows...
  Chunk 10: 2,000,000 rows processed...
  Chunk 20: 4,000,000 rows processed...
  Chunk 30: 6,000,000 rows processed...
  Chunk 40: 8,000,000 rows processed...
  Chunk 50: 10,000,000 rows processed...
  Chunk 60: 12,000,000 rows processed...
  Chunk 70: 14,000,000 rows processed...
  Chunk 80: 16,000,000 rows processed...
  Chunk 90: 18,000,000 rows processed...
  Chunk 100: 20,000,000 rows processed...
  Chunk 110: 22,000,000 rows processed...
  Chunk 120: 24,000,000 rows processed...

──────────────────────────────────────────────────────────────────────
YEAR 2020 SUMMARY
──────────────────────────────────────────────────────────────────────
Chunks processed:    129
Total rows kept:     12,050,951
Black applicants:    891

In [9]:
if len(results) > 0:
    # Create summary DataFrame
    summary_df = pd.DataFrame(results)
    
    # Display summary
    print("\n" + "="*70)
    print("SUMMARY ACROSS ALL YEARS")
    print("="*70 + "\n")
    
    # Format for display
    display_df = summary_df.copy()
    display_df['total_rows'] = display_df['total_rows'].apply(lambda x: f"{x:,}")
    display_df['black_count'] = display_df['black_count'].apply(lambda x: f"{x:,}")
    display_df['white_count'] = display_df['white_count'].apply(lambda x: f"{x:,}")
    display_df['black_pct'] = display_df['black_pct'].apply(lambda x: f"{x:.1f}%")
    display_df['approval_rate'] = display_df['approval_rate'].apply(lambda x: f"{x:.1f}%")
    
    print(display_df.to_string(index=False))
    
    # Save summary
    summary_file = PROCESSED_DATA_DIR / "cleaning_summary.csv"
    summary_df.to_csv(summary_file, index=False)
    summary_df.to_csv(Path("../outputs/tables/T00_cleaning_summary.csv"), index=False)
    print("✅ Also saved to outputs/tables/T00_cleaning_summary.csv")

    print(f"\n✅ Summary saved to: {summary_file}")
    
else:
    print("\n❌ ERROR: No years processed successfully")


SUMMARY ACROSS ALL YEARS

 year total_rows black_count white_count black_pct  approved_count  denied_count approval_rate
 2020 12,050,951     891,045  11,159,906      7.4%        10243794       1807157         85.0%
 2021 12,239,263   1,105,866  11,133,397      9.0%        10421528       1817735         85.1%
 2022  7,755,394     880,307   6,875,087     11.4%         6153575       1601819         79.3%
 2023  5,570,382     683,565   4,886,817     12.3%         4240582       1329800         76.1%
 2024  5,825,960     700,462   5,125,498     12.0%         4473145       1352815         76.8%
✅ Also saved to extreme_final_tables/T00_cleaning_summary.csv

✅ Summary saved to: ..\data\processed\cleaning_summary.csv


In [10]:
if len(results) > 0:
    # Calculate totals
    total_applications = summary_df["total_rows"].sum()
    total_black = summary_df["black_count"].sum()
    total_white = summary_df["white_count"].sum()
    total_approved = summary_df["approved_count"].sum()
    total_denied = summary_df["denied_count"].sum()
    
    print("\n" + "="*70)
    print("GRAND TOTALS (2020-2024)")
    print("="*70)
    print(f"\nTotal applications:  {total_applications:,}")
    print(f"")
    print(f"Black applications:  {total_black:,} ({total_black/total_applications*100:.1f}%)")
    print(f"White applications:  {total_white:,} ({total_white/total_applications*100:.1f}%)")
    print(f"")
    print(f"Total approved:      {total_approved:,} ({total_approved/total_applications*100:.1f}%)")
    print(f"Total denied:        {total_denied:,} ({total_denied/total_applications*100:.1f}%)")
    
    # Expected numbers check
    print("\n" + "─"*70)
    print("EXPECTED VALUES CHECK:")
    print("─"*70)
    
    if 43_000_000 < total_applications < 44_000_000:
        print("✅ Total applications: ~43.4M (correct)")
    else:
        print(f"⚠️  Total applications: {total_applications:,} (expected ~43.4M)")
    
    if 9 < total_black/total_applications*100 < 11:
        print("✅ Black share: ~9-10% (correct)")
    else:
        print(f"⚠️  Black share: {total_black/total_applications*100:.1f}% (expected ~9-10%)")
    
    if 75 < total_approved/total_applications*100 < 85:
        print("✅ Approval rate: ~75-85% (correct)")
    else:
        print(f"⚠️  Approval rate: {total_approved/total_applications*100:.1f}% (expected ~75-85%)")
    
    print("\n" + "="*70)
    print("✅ DATA CLEANING COMPLETE")
    print("="*70)
    print(f"\n📁 Panel files saved in: {PROCESSED_DATA_DIR}")
    print("\nYou can now proceed to Notebook 02: Descriptive Statistics")


GRAND TOTALS (2020-2024)

Total applications:  43,441,950

Black applications:  4,261,245 (9.8%)
White applications:  39,180,705 (90.2%)

Total approved:      35,532,624 (81.8%)
Total denied:        7,909,326 (18.2%)

──────────────────────────────────────────────────────────────────────
EXPECTED VALUES CHECK:
──────────────────────────────────────────────────────────────────────
✅ Total applications: ~43.4M (correct)
✅ Black share: ~9-10% (correct)
✅ Approval rate: ~75-85% (correct)

✅ DATA CLEANING COMPLETE

📁 Panel files saved in: ..\data\processed

You can now proceed to Notebook 02: Descriptive Statistics


In [11]:
# Load one panel file and show first few rows
sample_year = 2024
sample_file = PROCESSED_DATA_DIR / f"panel_{sample_year}.csv"

if sample_file.exists():
    print(f"Displaying first 10 rows from panel_{sample_year}.csv:\n")
    sample_df = pd.read_csv(sample_file, nrows=10)
    print(sample_df.to_string(index=False))
    
    print(f"\n\nColumn types:")
    print(sample_df.dtypes)
else:
    print(f"❌ File not found: {sample_file}")

Displaying first 10 rows from panel_2024.csv:

                 lei  year state_code  applicant_race_1  black  approved  income  loan_amount  property_value  interest_rate  rate_spread       ltv  dti  loan_purpose
WWB2V0FCW3A0EE3ZJN75  2024         CT               3.0      1         0    86.0       105000        455000.0            NaN          NaN 23.076923  NaN            31
WWB2V0FCW3A0EE3ZJN75  2024         CT               5.0      0         1   124.0        85000        695000.0          12.59         4.31 12.230216  NaN             2
WWB2V0FCW3A0EE3ZJN75  2024         PA               5.0      0         1   113.0       125000        865000.0           9.34         1.17 14.450867  NaN             2
WWB2V0FCW3A0EE3ZJN75  2024         PA               5.0      0         1    34.0        55000        225000.0          10.09         1.99 24.444444  NaN            31
WWB2V0FCW3A0EE3ZJN75  2024         CT               5.0      0         0    67.0       355000        425000.0         